In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import soundfile as sf
import torch
import torch.nn as nn


In [ ]:
import librosa
class DataProcessor:

    def __init__(self, class_name, input_path, df_meta_subset):
        self.class_name = class_name
        self.input_path = input_path
        self.df_meta_subset = df_meta_subset
        self.labels = np.empty(0, dtype=object)
        self.n_mels = 128
        self.n_fft = 2048
        self.hop_length = 512
    

    def _open_resample_melspec(self, file):
        data, samplerate = sf.read(f'{self.input_path}{file}')
        resampled_data = sp.signal.resample(data, samples)
        mel_spec = librosa.feature.melspectrogram(y=resampled_data, sr=samplerate, 
                                                  n_mels=self.n_mels, n_fft=self.n_fft, hop_length=self.hop_length)
        self.labels = np.append(self.labels, self.df_meta_subset[self.df_meta_subset["filename"] == file]['primary_label'].values[0])
        return mel_spec
    
    def process_files(self):
        self.mel_spec_array = np.array([self._open_resample_melspec(file) for file in self.df_meta_subset["filename"]])

    def save_arrays(self):
        np.save(f"{self.class_name}_mel_spec.npy", self.mel_spec_array)
        np.save(f"{self.class_name}_labels.npy", self.labels)

In [ ]:
def load_greyel_data(df_meta):
    input_path = 'data/train_audio/'
    class_type = 'greyel'
    df_meta_subset = df_meta[df_meta['filename'].str.contains(class_type)]

    greyel_processor = DataProcessor(class_type, input_path, df_meta_subset)
    greyel_processor.process_files()
    greyel_processor.save_arrays()

    size_mb = greyel_processor.mel_spec_array.nbytes / (1024 ** 2)
    print(f"{size_mb:.2f} MB")
    print(f"Shape of mel spectrogram array: {greyel_processor.mel_spec_array.shape}")
    print(f"Shape of labels array: {greyel_processor.labels.shape}")

In [ ]:

def load_grekis_data(df_meta):
    input_path = 'data/train_audio/'
    class_type = 'grekis'
    df_meta_subset = df_meta[df_meta['filename'].str.contains(class_type)]

    grekis_processor = DataProcessor(class_type, input_path, df_meta_subset)
    grekis_processor.process_files()
    grekis_processor.save_arrays()

    size_mb = grekis_processor.mel_spec_array.nbytes / (1024 ** 2)
    print(f"{size_mb:.2f} MB")
    print(f"Shape of mel spectrogram array: {grekis_processor.mel_spec_array.shape}")
    print(f"Shape of labels array: {grekis_processor.labels.shape}")

In [ ]:
def process_and_save_data():
    df_meta = pd.read_csv("data/train.csv")
    load_greyel_data(df_meta)
    load_grekis_data(df_meta)

In [ ]:
def load_data():
    data_1 = np.load("greyel_mel_spec.npy")
    y_1 = np.load("greyel_labels.npy", allow_pickle=True)

    data_2 = np.load("grekis_mel_spec.npy")
    y_2 = np.load("grekis_labels.npy", allow_pickle=True)

    data = np.vstack((data_1, data_2))
    y = np.append(y_1, y_2)
    return data, y

In [ ]:
def process_and_split_data(data, y, test_size=0.2, random_state=42, normalize=True):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from librosa.util import normalize as librosa_normalize
    from torch.utils.data import DataLoader, TensorDataset

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(data, y_encoded, test_size=test_size, random_state=random_state)

    if normalize:
        librosa_normalized_X_train = librosa_normalize(X_train, norm=np.inf)
        X_train = librosa_normalized_X_train
        librosa_normalized_X_test = librosa_normalize(X_test, norm=np.inf)
        X_test = librosa_normalized_X_test


    X_train = torch.tensor(X_train).float()
    y_train = torch.tensor(y_train).long()

    X_train = X_train.unsqueeze(1) # Add a channel dimension for CNN input

    train_ds = TensorDataset(X_train, y_train)

    batch_size = 20
    torch.manual_seed(1)
    train_dl = DataLoader(train_ds, batch_size, shuffle=True)

    X_test = torch.tensor(X_test).float().unsqueeze(1) # Add a channel dimension for CNN input
    y_test = torch.tensor(y_test).long()

    test_dl = DataLoader(TensorDataset(X_test, y_test), batch_size, shuffle=False)

    return train_dl, test_dl, label_encoder

In [ ]:
import math
class MelSpecCNN(nn.Module):
    
    
    def __init__(self, num_classes, H_in, W_in, kernel_size=5, channel_out_1=32, maxpool_kernel_size=2):
        super(MelSpecCNN, self).__init__()
        padding = kernel_size // 2
        stride = 1
        H_out = math.floor((H_in + 2 * padding - kernel_size) / stride) + 1
        W_out = math.floor((W_in + 2 * padding - kernel_size) / stride) + 1

        channel_out_2 = channel_out_1 * 2

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=channel_out_1, kernel_size=kernel_size, padding=padding)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=maxpool_kernel_size)
        self.conv2 = nn.Conv2d(in_channels=channel_out_1, out_channels=channel_out_2, kernel_size=kernel_size, padding=padding)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=maxpool_kernel_size)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(channel_out_2 * (H_out // (maxpool_kernel_size ** 2)) * (W_out // (maxpool_kernel_size ** 2)), 640)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.7)
        self.fc2 = nn.Linear(640, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:

def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            pred = model(x_batch.to(device))
            pred = pred.squeeze()  # Remove extra dimensions if necessary
            is_correct = (torch.argmax(pred, dim=1) == y_batch.to(device)).float()
            correct += is_correct.sum().cpu()
            total += y_batch.size(0)
    accuracy = correct / total
    return accuracy

def train(model, train_dl, test_dl, loss_fn, optimizer, device, num_epochs=20):
    torch.manual_seed(1)
    training_history = []
    validation_history = []
    for epoch in range(num_epochs):
        model.train()
        accuracy_hist_train = 0
        for x_batch, y_batch in train_dl:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch)
            pred = pred.squeeze()  # Remove extra dimensions if necessary
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
            accuracy_hist_train += is_correct.sum().cpu()
        accuracy_hist_train /= len(train_dl.dataset)
        validation_accuracy = evaluate(model, test_dl, device)
        training_history.append(accuracy_hist_train)
        validation_history.append(validation_accuracy)
        print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}  Validation Accuracy {validation_accuracy:.4f}')
    return training_history, validation_history

In [ ]:
from matplotlib import pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dl, test_dl, label_encoder = process_and_split_data(*load_data(), normalize=True)

H_in, W_in = train_dl.dataset.tensors[0].shape[2], train_dl.dataset.tensors[0].shape[3]
model = MelSpecCNN(num_classes=len(label_encoder.classes_), H_in=H_in, 
                   W_in=W_in, 
                   kernel_size=5, channel_out_1=16, 
                   maxpool_kernel_size=2)
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
training_history, validation_history = train(model, train_dl, test_dl, loss_fn, optimizer, device, num_epochs=20)

plt.plot(training_history, label='Training Accuracy', color='blue')
plt.plot(validation_history, label='Validation Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
# del model
# del optimizer
# # del X_train
# # del y_train
# # del X_test
# # del y_test
# torch.cuda.empty_cache()